# Profiling

Build a measurement workflow that moves from symptoms to evidence about bottlenecks.

## Objectives

Define a reproducible workload, collect appropriate evidence, and avoid perturbing or over-interpreting measurements.

## Background

Profilers add overhead and expose different layers of a system; useful conclusions require a stable baseline and a focused question.

## Prediction

The first workload will contain three explicit phases:

1. transform a sequence of integers using repeated arithmetic and bitwise operations;
2. build a histogram from the transformed values;
3. calculate a weighted checksum.

All three phases traverse the same number of elements, but the transformation performs several rounds of arithmetic per element. It should therefore account for the largest fraction of unprofiled runtime.

The histogram and checksum phases should remain visible but secondary. Their relative cost cannot be inferred from operation counts alone because Python integer arithmetic, list access, loop mechanics, and memory allocation all contribute to elapsed time.

Before collecting a profile, the workload must have:

- deterministic input and output;
- a bounded runtime;
- fixed CPU affinity;
- warm-up runs;
- repeated baseline measurements.

A deterministic function-level profiler such as `cProfile` should later attribute most cumulative time to the transformation phase. However, the profiled runtime should be longer than the unprofiled baseline because profiling records Python call events. The profile can therefore identify where observed runtime is attributed, but its timing should not be treated as an unperturbed measurement of production performance.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [ ]:
import os
from contextlib import contextmanager
from pprint import pprint

import pandas as pd

from common.benchmark import benchmark_callable


PROFILE_CPU = 15

print(f"Available CPUs: {sorted(os.sched_getaffinity(0))}")
print(f"Selected profiling CPU: {PROFILE_CPU}")

if PROFILE_CPU not in os.sched_getaffinity(0):
    raise RuntimeError(f"CPU {PROFILE_CPU} is not available to the current process")


@contextmanager
def pinned_to_cpu(cpu_id: int):
    original_affinity = os.sched_getaffinity(0)

    try:
        os.sched_setaffinity(0, {cpu_id})
        yield
    finally:
        os.sched_setaffinity(0, original_affinity)


with pinned_to_cpu(PROFILE_CPU):
    print(f"Temporary affinity: {sorted(os.sched_getaffinity(0))}")

Available CPUs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Selected profiling CPU: 15
Temporary affinity: [15]


### Controlled profiling workload

The workload below is intentionally divided into named Python functions. This will later allow a function-level profiler to attribute time to meaningful phases rather than to one monolithic function.

The input is constructed once and reused. Each measured invocation:

1. allocates and fills a transformed list;
2. builds a 256-bin histogram;
3. calculates a weighted checksum.

The returned values prevent the final results from being semantically unused and provide simple correctness checks.

This is not intended to model a useful application. It is a controlled workload for learning how baseline timing and profile evidence relate.

In [ ]:
ELEMENT_COUNT = 1_000_000
TRANSFORM_ROUNDS = 3

UINT32_MASK = (1 << 32) - 1
UINT64_MASK = (1 << 64) - 1

input_values = [
    ((index * 2_654_435_761) ^ (index >> 3)) & UINT32_MASK
    for index in range(ELEMENT_COUNT)
]

workload_configuration = {
    "element_count": ELEMENT_COUNT,
    "transform_rounds": TRANSFORM_ROUNDS,
    "histogram_bins": 256,
    "input_size_mib_estimate": (input_values.__sizeof__() / 1024**2),
}

pprint(workload_configuration)

{'element_count': 1000000,
 'histogram_bins': 256,
 'input_size_mib_estimate': 8.057319641113281,
 'transform_rounds': 3}


In [ ]:
def transform_values(
    values: list[int],
    rounds: int,
) -> list[int]:
    transformed = [0] * len(values)

    for index, value in enumerate(values):
        current = value

        for _ in range(rounds):
            current ^= current >> 16
            current = (current * 0x45D9F3B) & UINT32_MASK
            current ^= current >> 16

        transformed[index] = current

    return transformed


def build_low_byte_histogram(values: list[int]) -> list[int]:
    histogram = [0] * 256

    for value in values:
        histogram[value & 0xFF] += 1

    return histogram


def calculate_weighted_checksum(values: list[int]) -> int:
    checksum = 0

    for index, value in enumerate(values, start=1):
        checksum = (checksum + index * (value & 0xFFFF)) & UINT64_MASK

    return checksum


def profiling_workload() -> tuple[int, int, int]:
    transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    histogram = build_low_byte_histogram(transformed)
    checksum = calculate_weighted_checksum(transformed)

    return (
        checksum,
        sum(histogram),
        max(histogram),
    )

### Correctness and determinism check

Before timing the workload, run it twice and verify that:

- every transformed element is represented in the histogram;
- repeated executions produce the same result.

These executions also provide initial interpreter and memory-allocation warm-up, but they are not part of the measured baseline.

In [5]:
with pinned_to_cpu(PROFILE_CPU):
    first_result = profiling_workload()
    second_result = profiling_workload()

assert first_result == second_result
assert first_result[1] == ELEMENT_COUNT
assert 0 < first_result[2] <= ELEMENT_COUNT

correctness_result = {
    "checksum": first_result[0],
    "histogram_total": first_result[1],
    "largest_histogram_bin": first_result[2],
    "deterministic": first_result == second_result,
}

pprint(correctness_result)

{'checksum': 16378977129066911,
 'deterministic': True,
 'histogram_total': 1000000,
 'largest_histogram_bin': 4112}


### Unprofiled baseline

The baseline uses wall-clock time without an active profiler. All warm-up and measured iterations run while the notebook process is pinned to CPU 15.

Individual trials are retained because the distribution matters. A single minimum or mean cannot show scheduling noise, thermal effects, frequency changes, or occasional operating-system interference.

This baseline will later be compared with the runtime of the same workload under profiling.

In [ ]:
BASELINE_WARMUP_ITERATIONS = 1
BASELINE_ITERATIONS = 7

with pinned_to_cpu(PROFILE_CPU):
    baseline_result = benchmark_callable(
        profiling_workload,
        warmup_iterations=BASELINE_WARMUP_ITERATIONS,
        iterations=BASELINE_ITERATIONS,
    )

baseline_trials = pd.DataFrame(
    {
        "trial": range(1, baseline_result.iterations + 1),
        "wall_ms": [
            duration_ns / 1_000_000 for duration_ns in baseline_result.durations_ns
        ],
    }
)

baseline_trials.round(3)

,trial,wall_ms
0,1,470.828
1,2,470.727
2,3,470.911
3,4,469.696
4,5,470.071
5,6,468.926
6,7,469.325


In [ ]:
baseline_summary = pd.DataFrame(
    [
        {
            "iterations": baseline_result.iterations,
            "minimum_ms": baseline_result.minimum_ns / 1_000_000,
            "median_ms": baseline_result.median_ns / 1_000_000,
            "mean_ms": baseline_result.mean_ns / 1_000_000,
            "maximum_ms": baseline_result.maximum_ns / 1_000_000,
            "standard_deviation_ms": (
                baseline_result.standard_deviation_ns / 1_000_000
            ),
            "relative_standard_deviation_percent": (
                100 * baseline_result.standard_deviation_ns / baseline_result.mean_ns
            ),
        }
    ]
)

baseline_summary.round(3)

,iterations,minimum_ms,median_ms,mean_ms,maximum_ms,standard_deviation_ms,relative_standard_deviation_percent
0,7,468.926,470.071,470.069,470.911,0.729,0.155


## Observations

TODO: Record baseline timing, profiler overhead, and evidence from actual runs.

## Explanation

TODO: Explain the bottleneck using profile evidence and state the tool's limitations.

## Connection to LLMs

Profiling separates compute, memory, synchronization, input, and communication bottlenecks in LLM systems.

## Further Exploration

TODO: Form a single optimization hypothesis and define a before/after measurement.